In [1]:
!pip install requests

In [2]:
# Import requests library to call API (get data from internet)
import requests

# Import time library to add delay (avoid API blocking)
import time

# Create an empty list to store all records from all pages
records = []

# Base API URL (main endpoint)
base_url = "https://api.coingecko.com/api/v3/coins/markets"

# Loop through pages 1 to 5
for page in range(1, 6):

    # Parameters to send with API request
    params = {
        "vs_currency": "inr",              # Get prices in INR
        "order": "market_cap_desc",       # Sort by highest market cap
        "per_page": 250,                  # Get 250 records per page
        "page": page,                     # Current page number
        "sparkline": "false"              # Disable sparkline data
    }

    # Flag to check if request is successful
    success = False

    # Counter to track number of retry attempts
    attempts = 0

    # Retry loop: runs until success OR max 3 attempts reached
    while not success and attempts < 3:

        # Send GET request to API with parameters
        response = requests.get(base_url, params=params)

        # Check if request is successful (status code 200)
        if response.status_code == 200:

            # Convert response data into JSON format
            data = response.json()

            # Add all records from this page into main list
            records.extend(data)

            # Print success message
            print(f"Page {page} fetched ✅")

            # Mark as successful to stop retry loop
            success = True

        else:
            # Increase retry attempt count
            attempts += 1

            # Print error message with attempt number
            print(f"Error on page {page} ❌ (Attempt {attempts})")

            # Wait 5 seconds before retrying
            time.sleep(5)

    # Wait 2 seconds before moving to next page (avoid API rate limit)
    time.sleep(5)

# Print total number of records collected
print(f"\nTotal records collected: {len(records)}") 


Page 1 fetched ✅
Page 2 fetched ✅
Page 3 fetched ✅
Error on page 4 ❌ (Attempt 1)
Error on page 4 ❌ (Attempt 2)
Error on page 4 ❌ (Attempt 3)
Error on page 5 ❌ (Attempt 1)
Error on page 5 ❌ (Attempt 2)
Page 5 fetched ✅

Total records collected: 1000


In [14]:
# Convert records list into DataFrame
import pandas as pd
df = pd.DataFrame(records)
df.head()

,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,7029893.00,140724637054110,1,1.407246e+14,5.639521e+12,7094595.00,...,2.001572e+07,2.100000e+07,11187013.00,-37.16023,2025-10-06T18:57:42.558Z,3993.420000,1.759369e+05,2013-07-05T00:00:00.000Z,None,2026-04-14T14:02:07.379Z
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,222636.00,26871383861167,2,2.687138e+13,2.657246e+12,225729.00,...,1.206910e+08,NaN,431946.00,-48.45731,2025-08-24T19:21:03.333Z,28.130000,7.913204e+05,2015-10-20T00:00:00.000Z,"{'times': 41.34983766580072, 'currency': 'btc'...",2026-04-14T14:02:03.919Z
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,93.14,17200267954974,3,1.770945e+13,9.020261e+12,95.17,...,1.901414e+11,NaN,105.52,-11.75962,2025-01-25T16:51:57.989Z,36.860000,1.526076e+02,2015-03-02T00:00:00.000Z,None,2026-04-14T14:02:07.609Z
3,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,129.65,7965142008935,4,1.296952e+13,2.837850e+11,130.87,...,9.998569e+10,1.000000e+11,313.99,-58.70770,2025-07-21T15:02:00.541Z,0.159343,8.126696e+04,2013-08-16T00:00:00.000Z,None,2026-04-14T14:02:07.623Z
4,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,57937.00,7897882229222,5,7.897882e+12,1.404209e+11,58539.00,...,1.363563e+08,2.000000e+08,121422.00,-52.28422,2025-10-13T08:41:24.131Z,2.580000,2.241421e+06,2017-10-19T00:00:00.000Z,None,2026-04-14T14:02:07.138Z


In [15]:
# Select only required columns
filtered_df = df[[
    "id",
    "symbol",
    "name",
    "current_price",
    "market_cap",
    "market_cap_rank",
    "total_volume",
    "circulating_supply",
    "total_supply",
    "ath",
    "atl",
    "last_updated"
]]

# Convert 'last_updated' to only DATE (remove time)
filtered_df["last_updated"] = pd.to_datetime(filtered_df["last_updated"]).dt.date

# Show first 5 rows
print(filtered_df.head())

# Save to CSV file
filtered_df.to_csv("cryptocurrencies.csv", index=False)

print("Filtered data saved ✅")

            id symbol      name  current_price       market_cap  \
0      bitcoin    btc   Bitcoin     7029893.00  140724637054110   
1     ethereum    eth  Ethereum      222636.00   26871383861167   
2       tether   usdt    Tether          93.14   17200267954974   
3       ripple    xrp       XRP         129.65    7965142008935   
4  binancecoin    bnb       BNB       57937.00    7897882229222   

   market_cap_rank  total_volume  circulating_supply  total_supply  \
0                1  5.639521e+12        2.001572e+07  2.001572e+07   
1                2  2.657246e+12        1.206910e+08  1.206910e+08   
2                3  9.020261e+12        1.846744e+11  1.901414e+11   
3                4  2.837850e+11        6.140553e+10  9.998569e+10   
4                5  1.404209e+11        1.363563e+08  1.363563e+08   

           ath          atl last_updated  
0  11187013.00  3993.420000   2026-04-14  
1    431946.00    28.130000   2026-04-14  
2       105.52    36.860000   2026-04-14  
3   

C:\Users\HeY!\AppData\Local\Temp\ipykernel_21532\677050108.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["last_updated"] = pd.to_datetime(filtered_df["last_updated"]).dt.date


In [34]:
#Path C:\Users\HeY!\Project_1
import requests
import pandas as pd

# Change coin here
coin_id = "bitcoin"

# Dynamic URL 
url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=inr&days=365"

response = requests.get(url)
data = response.json()

# Convert to DataFrame
df_bitcoin = pd.DataFrame(data["prices"], columns=["timestamp", "price_inr"])

# Convert timestamp → date
df_bitcoin["date"] = pd.to_datetime( df_bitcoin["timestamp"], unit="ms").dt.date

# Add coin_id
df_bitcoin["coin_id"] = coin_id

# Final columns
df_bitcoin =  df_bitcoin[["coin_id", "date", "price_inr"]]

df_bitcoin.head()

,coin_id,date,price_inr
0,bitcoin,2025-04-15,7.271600e+06
1,bitcoin,2025-04-16,7.171573e+06
2,bitcoin,2025-04-17,7.201187e+06
3,bitcoin,2025-04-18,7.251703e+06
4,bitcoin,2025-04-19,7.209254e+06


In [35]:
#Path C:\Users\HeY!\Project_1
import requests
import pandas as pd

# Change coin here
coin_id = "ethereum"

# Dynamic URL 
url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=inr&days=365"

response = requests.get(url)
data = response.json()

# Convert to DataFrame
df_ethereum = pd.DataFrame(data["prices"], columns=["timestamp", "price_inr"])

# Convert timestamp → date
df_ethereum["date"] = pd.to_datetime( df_ethereum["timestamp"], unit="ms").dt.date

# Add coin_id
df_ethereum["coin_id"] = coin_id

# Final columns
df_ethereum =  df_ethereum[["coin_id", "date", "price_inr"]]

df_ethereum.head()

,coin_id,date,price_inr
0,ethereum,2025-04-15,139502.269181
1,ethereum,2025-04-16,136118.340300
2,ethereum,2025-04-17,135084.250075
3,ethereum,2025-04-18,135202.856452
4,ethereum,2025-04-19,135687.422766


In [36]:
#Path C:\Users\HeY!\Project_1
import requests
import pandas as pd

# Change coin here
coin_id = "tether"

# Dynamic URL 
url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=inr&days=365"

response = requests.get(url)
data = response.json()

# Convert to DataFrame
df_tether = pd.DataFrame(data["prices"], columns=["timestamp", "price_inr"])

# Convert timestamp → date
df_tether["date"] = pd.to_datetime( df_tether["timestamp"], unit="ms").dt.date

# Add coin_id
df_tether["coin_id"] = coin_id

# Final columns
df_tether =  df_tether[["coin_id", "date", "price_inr"]]

df_tether.head()

,coin_id,date,price_inr
0,tether,2025-04-15,86.024832
1,tether,2025-04-16,85.722129
2,tether,2025-04-17,85.620824
3,tether,2025-04-18,85.384061
4,tether,2025-04-19,85.370372


In [37]:
#concat coins
import pandas as pd

# Combine all dataframes
combined_df = pd.concat([df_bitcoin, df_ethereum, df_tether]).reset_index(drop=True)

# Save as CSV file
combined_df.to_csv("crypto_prices.csv", index=False)

print("crypto_prices.csv saved successfully ✅")

crypto_prices.csv saved successfully ✅


In [31]:
import pandas as pd

# Load data
oil_df = pd.read_csv("https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv")

# Convert to datetime
oil_df["Date"] = pd.to_datetime(oil_df["Date"])

# Filter from Jan 2020
oil_df = oil_df[oil_df["Date"] >= "2020-01-01"]

# Rename columns
oil_df = oil_df.rename(columns={"Date": "date", "Price": "price_usd"})

# ✅ Convert USD → INR
oil_df["price_inr"] = oil_df["price_usd"] * 83

# Keep required columns
oil_df = oil_df[["date", "price_inr"]]

# Save CSV
oil_df.to_csv("oil_prices.csv", index=False)

print("Oil data in INR saved ✅")

Oil data in INR saved ✅


In [35]:
import pandas as pd
oil_df=pd.read_csv(r'C:\Users\HeY!\Project_1\oil_prices.csv')
oil_df.tail()

,date,price_inr
1559,2026-03-30,8689.270000
1560,2026-03-31,8537.380000
1561,2026-04-01,8457.700000
1562,2026-04-02,9398.090000
1563,2026-04-06,9462.830000


In [1]:
import sys
print(sys.executable)

D:\guvi\python.exe


In [2]:
import sys
!{sys.executable} -m pip install yfinance --user

In [3]:
import yfinance as yf

In [4]:
import yfinance as yf
import pandas as pd

# Choose indices
tickers = ["^GSPC", "^IXIC", "^NSEI"]
start_date = "2020-01-01"
end_date = "2026-04-06"

# Download historical data
stocks_df = yf.download(tickers, start=start_date, end=end_date, group_by="ticker")
stocks_df.head()

[*********************100%***********************]  3 of 3 completed


Ticker             ^NSEI                                                      \
Price               Open          High           Low         Close    Volume   
Date                                                                           
2020-01-01  12202.150391  12222.200195  12165.299805  12182.500000  304100.0   
2020-01-02  12198.549805  12289.900391  12195.250000  12282.200195  407700.0   
2020-01-03  12261.099609  12265.599609  12191.349609  12226.650391  428800.0   
2020-01-06  12170.599609  12179.099609  11974.200195  11993.049805  396500.0   
2020-01-07  12079.099609  12152.150391  12005.349609  12052.950195  447800.0   

Ticker            ^IXIC                                                       \
Price              Open         High          Low        Close        Volume   
Date                                                                           
2020-01-01          NaN          NaN          NaN          NaN           NaN   
2020-01-02  9039.459961  9093.429688  9010.889648  9092.190430  2.862700e+09   
2020-01-03  8976.429688  9065.759766  8976.429688  9020.769531  2.586520e+09   
2020-01-06  8943.500000  9072.410156  8943.500000  9071.469727  2.810450e+09   
2020-01-07  9076.639648  9091.929688  9042.549805  9068.580078  2.381740e+09   

Ticker            ^GSPC                                                       
Price              Open         High          Low        Close        Volume  
Date                                                                          
2020-01-01          NaN          NaN          NaN          NaN           NaN  
2020-01-02  3244.669922  3258.139893  3235.530029  3257.850098  3.459930e+09  
2020-01-03  3226.360107  3246.149902  3222.340088  3234.850098  3.484700e+09  
2020-01-06  3217.550049  3246.840088  3214.639893  3246.280029  3.702460e+09  
2020-01-07  3241.860107  3244.909912  3232.429932  3237.179932  3.435910e+09

In [9]:
stock_1 = stocks_df['^GSPC'].reset_index()

# Add ticker column
stock_1["ticker"] = "^GSPC"

stock_1.head()

Price,Date,Open,High,Low,Close,Volume,ticker
0,2020-01-01,NaN,NaN,NaN,NaN,NaN,^GSPC
1,2020-01-02,3244.669922,3258.139893,3235.530029,3257.850098,3.459930e+09,^GSPC
2,2020-01-03,3226.360107,3246.149902,3222.340088,3234.850098,3.484700e+09,^GSPC
3,2020-01-06,3217.550049,3246.840088,3214.639893,3246.280029,3.702460e+09,^GSPC
4,2020-01-07,3241.860107,3244.909912,3232.429932,3237.179932,3.435910e+09,^GSPC


In [10]:
stock_2 = stocks_df['^IXIC'].reset_index()

# Add ticker column
stock_2["ticker"] = "^IXIC"

stock_2.head()

Price,Date,Open,High,Low,Close,Volume,ticker
0,2020-01-01,NaN,NaN,NaN,NaN,NaN,^IXIC
1,2020-01-02,9039.459961,9093.429688,9010.889648,9092.190430,2.862700e+09,^IXIC
2,2020-01-03,8976.429688,9065.759766,8976.429688,9020.769531,2.586520e+09,^IXIC
3,2020-01-06,8943.500000,9072.410156,8943.500000,9071.469727,2.810450e+09,^IXIC
4,2020-01-07,9076.639648,9091.929688,9042.549805,9068.580078,2.381740e+09,^IXIC


In [12]:
stock_3 = stocks_df['^NSEI'].reset_index()

# Add ticker column
stock_3["ticker"] = "^NSEI"

stock_3.head()

Price,Date,Open,High,Low,Close,Volume,ticker
0,2020-01-01,12202.150391,12222.200195,12165.299805,12182.500000,304100.0,^NSEI
1,2020-01-02,12198.549805,12289.900391,12195.250000,12282.200195,407700.0,^NSEI
2,2020-01-03,12261.099609,12265.599609,12191.349609,12226.650391,428800.0,^NSEI
3,2020-01-06,12170.599609,12179.099609,11974.200195,11993.049805,396500.0,^NSEI
4,2020-01-07,12079.099609,12152.150391,12005.349609,12052.950195,447800.0,^NSEI


In [15]:
final_stocks = pd.concat([stock_1, stock_2, stock_3], ignore_index=True)
final_stocks.to_csv("stock_prices.csv", index=False)
final_stocks

Price,Date,Open,High,Low,Close,Volume,ticker
0,2020-01-01,NaN,NaN,NaN,NaN,NaN,^GSPC
1,2020-01-02,3244.669922,3258.139893,3235.530029,3257.850098,3.459930e+09,^GSPC
2,2020-01-03,3226.360107,3246.149902,3222.340088,3234.850098,3.484700e+09,^GSPC
3,2020-01-06,3217.550049,3246.840088,3214.639893,3246.280029,3.702460e+09,^GSPC
4,2020-01-07,3241.860107,3244.909912,3232.429932,3237.179932,3.435910e+09,^GSPC
...,...,...,...,...,...,...,...
4861,2026-03-27,23173.550781,23186.099609,22804.550781,22819.599609,6.126000e+05,^NSEI
4862,2026-03-30,22549.650391,22714.099609,22283.849609,22331.400391,6.986000e+05,^NSEI
4863,2026-03-31,NaN,NaN,NaN,NaN,NaN,^NSEI
4864,2026-04-01,22899.000000,22941.300781,22618.599609,22679.400391,5.180000e+05,^NSEI


In [4]:
!pip install mysql-connector-python pandas

In [5]:
# Import MySQL connector (connect Python with MySQL)
import mysql.connector

# Step 1: Connect to MySQL server
conn = mysql.connector.connect(
    host="localhost",   # your system
    user="root",        # MySQL username
    password="12345"    # your MySQL password
)

# Step 2: Create cursor (used to run SQL queries)
cursor = conn.cursor()

print("Connected successfully ✅")

Connected successfully ✅


In [6]:
# Create database
cursor.execute("CREATE DATABASE IF NOT EXISTS market_analysis")

# Select (use) that database
cursor.execute("USE market_analysis")

print("Database ready ✅")

Database ready ✅


In [3]:
# Create cryptocurrencies table
cursor.execute("""
CREATE TABLE IF NOT EXISTS cryptocurrencies (
    id VARCHAR(50) PRIMARY KEY,
    symbol VARCHAR(10),
    name VARCHAR(100),
    current_price DECIMAL(18,6),
    market_cap BIGINT,
    market_cap_rank INT,
    total_volume BIGINT,
    circulating_supply DECIMAL(20,6),
    total_supply DECIMAL(20,6),
    ath DECIMAL(18,6),
    atl DECIMAL(18,6),
    date DATE
)
""")

print("Table created ✅")

Table created ✅


In [4]:
# Create crypto_prices table
cursor.execute("""
CREATE TABLE IF NOT EXISTS crypto_prices (
    coin_id VARCHAR(50),      -- coin name like bitcoin, ethereum
    date DATE,                -- date of price
    price_inr DECIMAL(18,6)   -- price in INR
)
""")

print("crypto_prices table created ✅")

crypto_prices table created ✅


In [5]:
# Create oil_prices table
cursor.execute("""
CREATE TABLE IF NOT EXISTS oil_prices (
    date DATE PRIMARY KEY,    -- unique date
    price_inr DECIMAL(18,6)   -- oil price in INR
)
""")

print("oil_prices table created ✅")

oil_prices table created ✅


In [6]:
# Create stock_prices table
cursor.execute("""
CREATE TABLE IF NOT EXISTS stock_prices (
    date DATE,                -- trading date
    open DECIMAL(18,6),       -- opening price
    high DECIMAL(18,6),       -- highest price
    low DECIMAL(18,6),        -- lowest price
    close DECIMAL(18,6),      -- closing price
    volume BIGINT,            -- trading volume
    ticker VARCHAR(20)        -- stock index (^GSPC, ^IXIC, ^NSEI)
)
""")

print("stock_prices table created ✅")

stock_prices table created ✅


In [7]:
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table)

('crypto_prices',)
('cryptocurrencies',)
('oil_prices',)
('stock_prices',)


In [9]:
import pandas as pd
df = pd.read_csv("cryptocurrencies.csv")

# Rename column to match SQL table
df.rename(columns={"last_updated": "date"}, inplace=True)

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("cryptocurrencies.csv")

# Rename column
df.rename(columns={"last_updated": "date"}, inplace=True)

# 🔥 Convert ALL nan / NaN / empty → None
df = df.replace({np.nan: None, "nan": None, "": None})

In [13]:
df = df[
    [
        "id", "symbol", "name", "current_price", "market_cap",
        "market_cap_rank", "total_volume", "circulating_supply",
        "total_supply", "ath", "atl", "date"
    ]
]

In [14]:
for _, row in df.iterrows():
    values = tuple(None if str(x).lower() == "nan" else x for x in row)

    cursor.execute("""
        INSERT IGNORE INTO cryptocurrencies 
        (id, symbol, name, current_price, market_cap, market_cap_rank,
         total_volume, circulating_supply, total_supply, ath, atl, date)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, values)

conn.commit()

print("Inserted successfully ✅")

Inserted successfully ✅


In [15]:
cursor.execute("SELECT COUNT(*) FROM cryptocurrencies")
print(cursor.fetchall())

[(1000,)]


In [16]:
import pandas as pd
import numpy as np

# Step 1: Load CSV file
df = pd.read_csv("crypto_prices.csv")

# Step 2: Clean data (VERY IMPORTANT)
df = df.replace({np.nan: None, "nan": None, "": None})

# Step 3: Ensure correct column order
df = df[["coin_id", "date", "price_inr"]]

# Step 4: Insert into MySQL
data = []

for _, row in df.iterrows():
    # Convert any 'nan' string to None
    values = tuple(None if str(x).lower() == "nan" else x for x in row)
    data.append(values)

cursor.executemany("""
    INSERT INTO crypto_prices (coin_id, date, price_inr)
    VALUES (%s, %s, %s)
""", data)

# Step 5: Save changes
conn.commit()

print("crypto_prices inserted ✅")

crypto_prices inserted ✅


In [17]:
import pandas as pd
import numpy as np

# Step 1: Load CSV
df = pd.read_csv("oil_prices.csv")

# Step 2: Clean data
df = df.replace({np.nan: None, "nan": None, "": None})

# Step 3: Ensure column order
df = df[["date", "price_inr"]]

# Step 4: Insert
data = []

for _, row in df.iterrows():
    values = tuple(None if str(x).lower() == "nan" else x for x in row)
    data.append(values)

cursor.executemany("""
    INSERT IGNORE INTO oil_prices (date, price_inr)
    VALUES (%s, %s)
""", data)

# Step 5: Commit
conn.commit()

print("oil_prices inserted ✅")

oil_prices inserted ✅


In [18]:
import pandas as pd

df = pd.read_csv("stock_prices.csv")

# Fix column names
df.rename(columns={
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
}, inplace=True)

df.to_csv("stock_prices.csv", index=False)

print("Columns fixed ✅")

Columns fixed ✅


In [19]:
import pandas as pd
import numpy as np

# Step 1: Load CSV
df = pd.read_csv("stock_prices.csv")

# Step 2: Clean data
df = df.replace({np.nan: None, "nan": None, "": None})

# Step 3: Ensure correct column order
df = df[["date", "open", "high", "low", "close", "volume", "ticker"]]

# Step 4: Insert
data = []

for _, row in df.iterrows():
    values = tuple(None if str(x).lower() == "nan" else x for x in row)
    data.append(values)

cursor.executemany("""
    INSERT INTO stock_prices (date, open, high, low, close, volume, ticker)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", data)

# Step 5: Commit
conn.commit()

print("stock_prices inserted ✅")

stock_prices inserted ✅


In [20]:
cursor.execute("SELECT COUNT(*) FROM crypto_prices")
print("crypto_prices:", cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM oil_prices")
print("oil_prices:", cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM stock_prices")
print("stock_prices:", cursor.fetchall())

crypto_prices: [(1098,)]
oil_prices: [(1564,)]
stock_prices: [(4866,)]


In [ ]:
#1️⃣ cryptocurrencies

In [25]:
#Top 3 cryptocurrencies by market cap
cursor.execute("""
SELECT name, market_cap
FROM cryptocurrencies
ORDER BY market_cap DESC
LIMIT 3
""")

for row in cursor.fetchall():
    print(row)

('Bitcoin', 140724637054110)
('Ethereum', 26871383861167)
('Tether', 17200267954974)


In [31]:
#List all coins where circulating supply exceeds 90% of total supply.
cursor.execute("""
SELECT name, circulating_supply, total_supply
FROM cryptocurrencies
WHERE circulating_supply >= 0.9 * total_supply
""")

# Fetch and print results
for row in cursor.fetchall():
    print(row)

('1INCH', Decimal('1404568058.661079'), Decimal('1499999999.997000'))
('4', Decimal('1000000000.000000'), Decimal('1000000000.000000'))
('69420', Decimal('62502297000.000000'), Decimal('69420000000.000000'))
('A7A5', Decimal('39202265328.270740'), Decimal('39202265328.270740'))
('Aave', Decimal('15164786.532628'), Decimal('16000000.000000'))
('Act I The AI Prophecy', Decimal('948241537.069214'), Decimal('948241537.069214'))
('heyAura', Decimal('147900000.000000'), Decimal('150000000.000000'))
('Aegis YUSD', Decimal('36512404.354581'), Decimal('36512404.354581'))
('Aergo', Decimal('472499995.768921'), Decimal('500000000.000000'))
('affine', Decimal('2891495.101057'), Decimal('2891495.101057'))
('AgentFun.AI', Decimal('100000000.000000'), Decimal('100000000.000000'))
('AUSD', Decimal('216456853.000000'), Decimal('216456853.000000'))
('AhaToken', Decimal('7257848898.024749'), Decimal('7673122637.674749'))
('AI Companions', Decimal('1000000000.000000'), Decimal('1000000000.000000'))
('AI R

In [34]:
#3rd query result (coins within 10% of ATH)
cursor.execute("""
SELECT name, current_price, ath
FROM cryptocurrencies
WHERE current_price >= 0.9 * ath
""")

for row in cursor.fetchall():
    print(row)

('ADI', Decimal('401.650000'), Decimal('421.780000'))
('AUSD', Decimal('93.090000'), Decimal('95.190000'))
('AIntivirus', Decimal('2.860000'), Decimal('2.890000'))
('Alloy Tether', Decimal('93.000000'), Decimal('95.210000'))
('Alphabet Class A (Ondo Tokenized Stock)', Decimal('30431.000000'), Decimal('31792.000000'))
('Altura Vault Tokens', Decimal('98.030000'), Decimal('101.060000'))
('Anzen USDz', Decimal('92.390000'), Decimal('94.820000'))
('APES', Decimal('7.060000'), Decimal('7.130000'))
('Apollo Diversified Credit Securitize Fund', Decimal('101746.000000'), Decimal('104019.000000'))
('Aster USDF', Decimal('93.000000'), Decimal('95.080000'))
('Avant USD', Decimal('93.060000'), Decimal('95.130000'))
('Balsa MM Fund', Decimal('7.630000'), Decimal('7.790000'))
('BFUSD', Decimal('93.070000'), Decimal('95.160000'))
('bitcastle Token', Decimal('18.680000'), Decimal('18.940000'))
('BlackRock USD Institutional Digital Liquidity Fund', Decimal('93.080000'), Decimal('95.200000'))
('Blockcha

In [36]:
#Find the average market cap rank of coins with volume above $1B
cursor.execute("""
SELECT name, total_volume
FROM cryptocurrencies
ORDER BY total_volume DESC
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)


('Tether', 9020260876145)
('Bitcoin', 5639520659546)
('Ethereum', 2657245525720)
('USDC', 1926199489804)
('Solana', 465637572196)
('XRP', 283785036162)
('Dogecoin', 199425581284)
('USD1', 147918877099)
('BNB', 140420868514)
('RaveDAO', 68357774705)


In [39]:
# 5.Get the most recently updated coin.
cursor.execute("""
SELECT name, current_price, date
FROM cryptocurrencies
WHERE date = (SELECT MAX(date) FROM cryptocurrencies)
""")

for row in cursor.fetchall():
    print(row)

('0x Protocol', Decimal('9.990000'), datetime.date(2026, 4, 14))
('1INCH', Decimal('8.940000'), datetime.date(2026, 4, 14))
('4', Decimal('1.076000'), datetime.date(2026, 4, 14))
('A7A5', Decimal('1.160000'), datetime.date(2026, 4, 14))
('Aave', Decimal('9412.770000'), datetime.date(2026, 4, 14))
('Abelian', Decimal('9.040000'), datetime.date(2026, 4, 14))
('Achain', Decimal('1.210000'), datetime.date(2026, 4, 14))
('Across Protocol', Decimal('4.000000'), datetime.date(2026, 4, 14))
('Act I The AI Prophecy', Decimal('1.220000'), datetime.date(2026, 4, 14))
('heyAura', Decimal('7.090000'), datetime.date(2026, 4, 14))
('ADI', Decimal('401.650000'), datetime.date(2026, 4, 14))
('Aegis YUSD', Decimal('92.950000'), datetime.date(2026, 4, 14))
('aelf', Decimal('7.410000'), datetime.date(2026, 4, 14))
('Aergo', Decimal('5.110000'), datetime.date(2026, 4, 14))
('Aerodrome Finance', Decimal('36.010000'), datetime.date(2026, 4, 14))
('Aethir', Decimal('0.576438'), datetime.date(2026, 4, 14))
('a

In [ ]:
#2️⃣ crypto_prices (daily prices of top coins)

In [42]:
#Find the highest daily price of Bitcoin in the last 365 days.
cursor.execute("""
SELECT 
    coin_id,
    MAX(price_inr) AS highest_price
FROM crypto_prices
WHERE coin_id = 'bitcoin'
""")

for row in cursor.fetchall():
    print(row)

('bitcoin', Decimal('11070472.246733'))


In [43]:
#Calculate the average daily price of Ethereum in the past 1 year.
cursor.execute("""
SELECT 
    coin_id,
    ROUND(AVG(price_inr), 2) AS avg_price
FROM crypto_prices
WHERE coin_id = 'ethereum'
""")

for row in cursor.fetchall():
    print(row)

('ethereum', Decimal('267001.88'))


In [48]:
#Show the daily price trend of Bitcoin in January 2025. (or change the month and year according you your data)
cursor.execute("""
SELECT date, price_inr
FROM crypto_prices
WHERE coin_id = 'bitcoin'

-- Last 30 days automatically
AND date >= DATE_SUB((SELECT MAX(date) FROM crypto_prices), INTERVAL 30 DAY)

ORDER BY date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2026, 3, 15), Decimal('6593624.616147'))
(datetime.date(2026, 3, 16), Decimal('6727548.658016'))
(datetime.date(2026, 3, 17), Decimal('6906940.623961'))
(datetime.date(2026, 3, 18), Decimal('6829860.404015'))
(datetime.date(2026, 3, 19), Decimal('6639390.135900'))
(datetime.date(2026, 3, 20), Decimal('6504037.303026'))
(datetime.date(2026, 3, 21), Decimal('6635372.108167'))
(datetime.date(2026, 3, 22), Decimal('6461678.631620'))
(datetime.date(2026, 3, 23), Decimal('6364717.089252'))
(datetime.date(2026, 3, 24), Decimal('6614293.785504'))
(datetime.date(2026, 3, 25), Decimal('6649392.307316'))
(datetime.date(2026, 3, 26), Decimal('6712173.732416'))
(datetime.date(2026, 3, 27), Decimal('6487672.474785'))
(datetime.date(2026, 3, 28), Decimal('6291490.149057'))
(datetime.date(2026, 3, 29), Decimal('6286074.723244'))
(datetime.date(2026, 3, 30), Decimal('6252812.763420'))
(datetime.date(2026, 3, 31), Decimal('6293233.346703'))
(datetime.date(2026, 4, 1), Decimal('6378628.791

In [49]:
#Find the coin with the highest average price over 1 year.
cursor.execute("""
SELECT coin_id, AVG(price_inr) AS avg_price
FROM crypto_prices

-- Group data for each coin
GROUP BY coin_id

-- Sort by highest average price
ORDER BY avg_price DESC

-- Take top coin
LIMIT 1
""")

for row in cursor.fetchall():
    print(row)

('bitcoin', Decimal('8566681.4082241366'))


In [52]:
#Get the % change in Bitcoin’s price between Sep 2024 and Sep 2025
cursor.execute("""
SELECT 
(
    MAX(price_inr) - MIN(price_inr)
) / MIN(price_inr) * 100 AS percentage_change

FROM crypto_prices
WHERE coin_id = 'bitcoin'
""")

for row in cursor.fetchall():
    print(row)

(Decimal('95.0182427944'),)


In [ ]:
#3️⃣ oil_prices

In [54]:
#Find the highest oil price in the last 5 years.
cursor.execute("""
SELECT date, price_inr
FROM oil_prices
ORDER BY price_inr DESC
LIMIT 1
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2022, 3, 8), Decimal('10262.120000'))


In [56]:
#Get the average oil price per year.
cursor.execute("""
SELECT 
    YEAR(date) AS year,
    ROUND(AVG(price_inr), 2) AS avg_price
FROM oil_prices
GROUP BY YEAR(date)
ORDER BY year
""")

for row in cursor.fetchall():
    print(row)

(2020, Decimal('3250.32'))
(2021, Decimal('5655.21'))
(2022, Decimal('7876.94'))
(2023, Decimal('6438.85'))
(2024, Decimal('6360.48'))
(2025, Decimal('5427.21'))
(2026, Decimal('6180.88'))


In [59]:
#Show oil prices during COVID crash (March–April 2020).
cursor.execute("""
SELECT date, price_inr
FROM oil_prices

-- Filter COVID crash period
WHERE date BETWEEN '2020-03-01' AND '2020-04-30'

-- Sort by date
ORDER BY date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2020, 3, 2), Decimal('3882.740000'))
(datetime.date(2020, 3, 3), Decimal('3923.410000'))
(datetime.date(2020, 3, 4), Decimal('3882.740000'))
(datetime.date(2020, 3, 5), Decimal('3809.700000'))
(datetime.date(2020, 3, 6), Decimal('3414.620000'))
(datetime.date(2020, 3, 9), Decimal('2577.150000'))
(datetime.date(2020, 3, 10), Decimal('2861.010000'))
(datetime.date(2020, 3, 11), Decimal('2749.790000'))
(datetime.date(2020, 3, 12), Decimal('2619.480000'))
(datetime.date(2020, 3, 13), Decimal('2632.760000'))
(datetime.date(2020, 3, 16), Decimal('2403.680000'))
(datetime.date(2020, 3, 17), Decimal('2237.680000'))
(datetime.date(2020, 3, 18), Decimal('1699.840000'))
(datetime.date(2020, 3, 19), Decimal('2082.470000'))
(datetime.date(2020, 3, 20), Decimal('1616.840000'))
(datetime.date(2020, 3, 23), Decimal('1936.390000'))
(datetime.date(2020, 3, 24), Decimal('1745.490000'))
(datetime.date(2020, 3, 25), Decimal('1722.250000'))
(datetime.date(2020, 3, 26), Decimal('1377.800000'))

In [65]:
#Find the lowest price of oil in the last 10 years.
cursor.execute("""
SELECT date, price_inr
FROM oil_prices
WHERE price_inr > 0
ORDER BY price_inr ASC
LIMIT 1
""")

print(cursor.fetchall())

[(datetime.date(2020, 4, 21), Decimal('739.530000'))]


In [66]:
#Calculate the volatility of oil prices (max-min difference per year).
cursor.execute("""
SELECT 
    YEAR(date) AS year,

    -- Highest price in that year
    MAX(price_inr) AS max_price,

    -- Lowest price in that year
    MIN(price_inr) AS min_price,

    -- Volatility = max - min
    (MAX(price_inr) - MIN(price_inr)) AS volatility

FROM oil_prices

-- Group data year-wise
GROUP BY YEAR(date)

-- Sort by year
ORDER BY year
""")

for row in cursor.fetchall():
    print(row)

(2020, Decimal('5251.410000'), Decimal('-3069.340000'), Decimal('8320.750000'))
(2021, Decimal('7108.120000'), Decimal('3940.010000'), Decimal('3168.110000'))
(2022, Decimal('10262.120000'), Decimal('5897.150000'), Decimal('4364.970000'))
(2023, Decimal('7774.610000'), Decimal('5528.630000'), Decimal('2245.980000'))
(2024, Decimal('7278.270000'), Decimal('5538.590000'), Decimal('1739.680000'))
(2025, Decimal('6700.590000'), Decimal('4601.520000'), Decimal('2099.070000'))
(2026, Decimal('9462.830000'), Decimal('4648.830000'), Decimal('4814.000000'))


In [ ]:
#4️⃣ stock_prices

In [74]:
#Get all stock prices for a given ticker
cursor.execute("""
SELECT 
    ticker,          -- stock/index name
    Date,            -- date of trading
    Close            -- closing price
FROM stock_prices

-- remove missing values (market closed days)
WHERE Close IS NOT NULL

-- sort properly: first by ticker, then by date
ORDER BY ticker, Date
""")

# Fetch and print results
for row in cursor.fetchall():
    print(row)

('^GSPC', datetime.date(2020, 1, 2), Decimal('3257.850098'))
('^GSPC', datetime.date(2020, 1, 3), Decimal('3234.850098'))
('^GSPC', datetime.date(2020, 1, 6), Decimal('3246.280029'))
('^GSPC', datetime.date(2020, 1, 7), Decimal('3237.179932'))
('^GSPC', datetime.date(2020, 1, 8), Decimal('3253.050049'))
('^GSPC', datetime.date(2020, 1, 9), Decimal('3274.699951'))
('^GSPC', datetime.date(2020, 1, 10), Decimal('3265.350098'))
('^GSPC', datetime.date(2020, 1, 13), Decimal('3288.129883'))
('^GSPC', datetime.date(2020, 1, 14), Decimal('3283.149902'))
('^GSPC', datetime.date(2020, 1, 15), Decimal('3289.290039'))
('^GSPC', datetime.date(2020, 1, 16), Decimal('3316.810059'))
('^GSPC', datetime.date(2020, 1, 17), Decimal('3329.620117'))
('^GSPC', datetime.date(2020, 1, 21), Decimal('3320.790039'))
('^GSPC', datetime.date(2020, 1, 22), Decimal('3321.750000'))
('^GSPC', datetime.date(2020, 1, 23), Decimal('3325.540039'))
('^GSPC', datetime.date(2020, 1, 24), Decimal('3295.469971'))
('^GSPC', date

In [75]:
#Find the highest closing price for NASDAQ (^IXIC)
cursor.execute("""
SELECT Date, Close
FROM stock_prices
WHERE ticker = '^IXIC'
AND Close IS NOT NULL
ORDER BY Close DESC
LIMIT 1
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 10, 29), Decimal('23958.470703'))


In [76]:
#List top 5 days with highest price difference (high - low) for S&P 500 (^GSPC)
cursor.execute("""
SELECT 
    Date,

    -- Daily highest price
    High,

    -- Daily lowest price
    Low,

    -- Difference (volatility of that day)
    (High - Low) AS price_difference

FROM stock_prices

-- Filter S&P 500
WHERE ticker = '^GSPC'

-- Remove missing values
AND High IS NOT NULL
AND Low IS NOT NULL

-- Sort by highest difference
ORDER BY price_difference DESC

-- Take top 5 days
LIMIT 5
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 4, 9), Decimal('5481.339844'), Decimal('4948.430176'), Decimal('532.909668'))
(datetime.date(2025, 4, 7), Decimal('5246.569824'), Decimal('4835.040039'), Decimal('411.529785'))
(datetime.date(2025, 4, 8), Decimal('5267.470215'), Decimal('4910.419922'), Decimal('357.050293'))
(datetime.date(2025, 4, 10), Decimal('5353.149902'), Decimal('5115.270020'), Decimal('237.879882'))
(datetime.date(2025, 11, 20), Decimal('6770.350098'), Decimal('6534.049805'), Decimal('236.300293'))


In [77]:
#Get monthly average closing price for each ticker
cursor.execute("""
SELECT 
    ticker,

    -- Extract year
    YEAR(Date) AS year,

    -- Extract month
    MONTH(Date) AS month,

    -- Average closing price
    AVG(Close) AS avg_close

FROM stock_prices

-- Remove missing values
WHERE Close IS NOT NULL

-- Group by ticker + year + month
GROUP BY ticker, YEAR(Date), MONTH(Date)

-- Sort properly
ORDER BY ticker, year, month
""")

for row in cursor.fetchall():
    print(row)

('^GSPC', 2020, 1, Decimal('3278.2028576667'))
('^GSPC', 2020, 2, Decimal('3277.3141832632'))
('^GSPC', 2020, 3, Decimal('2652.3936323636'))
('^GSPC', 2020, 4, Decimal('2761.9752255714'))
('^GSPC', 2020, 5, Decimal('2919.6084838000'))
('^GSPC', 2020, 6, Decimal('3104.6609330909'))
('^GSPC', 2020, 7, Decimal('3207.6190962273'))
('^GSPC', 2020, 8, Decimal('3391.7100190952'))
('^GSPC', 2020, 9, Decimal('3365.5166713810'))
('^GSPC', 2020, 10, Decimal('3418.6999956818'))
('^GSPC', 2020, 11, Decimal('3548.9924803500'))
('^GSPC', 2020, 12, Decimal('3695.3100141818'))
('^GSPC', 2021, 1, Decimal('3793.7484323158'))
('^GSPC', 2021, 2, Decimal('3883.4321160526'))
('^GSPC', 2021, 3, Decimal('3910.5082796522'))
('^GSPC', 2021, 4, Decimal('4141.1761997619'))
('^GSPC', 2021, 5, Decimal('4167.8495361500'))
('^GSPC', 2021, 6, Decimal('4238.4895463182'))
('^GSPC', 2021, 7, Decimal('4363.7127976190'))
('^GSPC', 2021, 8, Decimal('4453.9659312727'))
('^GSPC', 2021, 9, Decimal('4445.5433173810'))
('^GSPC', 

In [6]:
#Get average trading volume of NSEI in 2024
cursor.execute("""
SELECT AVG(volume) AS avg_volume
FROM stock_prices

-- Filter only NSEI index
WHERE ticker = '^NSEI'

-- Filter year 2024 data
AND YEAR(date) = 2024
""")

for row in cursor.fetchall():
    print(row)

(Decimal('313560.1626'),)


In [ ]:
#🔗 Join Queries (Cross-Market Analysis)

In [9]:
#Compare Bitcoin vs Oil average price in 2025.
cursor.execute("""
SELECT 
    ROUND(AVG(cp.price_inr),2) AS bitcoin_avg,
    ROUND(AVG(op.price_inr),2) AS oil_avg
FROM crypto_prices cp
JOIN oil_prices op 
ON cp.date = op.date
WHERE cp.coin_id = 'bitcoin'
AND YEAR(cp.date) = 2025
""")

print(cursor.fetchall())

[(Decimal('9245356.33'), Decimal('5253.94'))]


In [10]:
#Check if Bitcoin moves with S&P 500 (correlation idea).
cursor.execute("""
SELECT 
    cp.date,
    cp.price_inr AS bitcoin_price,
    sp.close AS sp500_price

FROM crypto_prices cp

-- Join with stock data using date
JOIN stock_prices sp 
ON cp.date = sp.date

-- Filter Bitcoin
WHERE cp.coin_id = 'bitcoin'

-- Filter S&P 500
AND sp.ticker = '^GSPC'

-- Sort for trend
ORDER BY cp.date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 4, 15), Decimal('7271599.950977'), Decimal('5396.629883'))
(datetime.date(2025, 4, 16), Decimal('7171572.949307'), Decimal('5275.700195'))
(datetime.date(2025, 4, 17), Decimal('7201187.297551'), Decimal('5282.700195'))
(datetime.date(2025, 4, 21), Decimal('7263849.726469'), Decimal('5158.200195'))
(datetime.date(2025, 4, 22), Decimal('7447341.987487'), Decimal('5287.759766'))
(datetime.date(2025, 4, 23), Decimal('7975735.892698'), Decimal('5375.859863'))
(datetime.date(2025, 4, 24), Decimal('8003280.213214'), Decimal('5484.770020'))
(datetime.date(2025, 4, 25), Decimal('8000169.783417'), Decimal('5525.209961'))
(datetime.date(2025, 4, 28), Decimal('8009924.381134'), Decimal('5528.750000'))
(datetime.date(2025, 4, 29), Decimal('8092853.390806'), Decimal('5560.830078'))
(datetime.date(2025, 4, 30), Decimal('8027696.597799'), Decimal('5569.060059'))
(datetime.date(2025, 5, 1), Decimal('7968688.759781'), Decimal('5604.140137'))
(datetime.date(2025, 5, 2), Decimal('8167

In [12]:
#Compare Ethereum and NASDAQ daily prices for 2025.
cursor.execute("""
SELECT 
    cp.date,
    cp.price_inr AS ethereum_price,
    sp.close AS nasdaq_price

FROM crypto_prices cp

-- Join using same date
JOIN stock_prices sp 
ON cp.date = sp.date

-- Filter Ethereum
WHERE cp.coin_id = 'ethereum'

-- Filter NASDAQ
AND sp.ticker = '^IXIC'

-- Filter year (change if needed)
AND YEAR(cp.date) = 2025

-- Sort for trend
ORDER BY cp.date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 4, 15), Decimal('139502.269181'), Decimal('16823.169922'))
(datetime.date(2025, 4, 16), Decimal('136118.340300'), Decimal('16307.160156'))
(datetime.date(2025, 4, 17), Decimal('135084.250075'), Decimal('16286.450195'))
(datetime.date(2025, 4, 21), Decimal('135372.221202'), Decimal('15870.900391'))
(datetime.date(2025, 4, 22), Decimal('134334.008185'), Decimal('16300.419922'))
(datetime.date(2025, 4, 23), Decimal('149984.444802'), Decimal('16708.050781'))
(datetime.date(2025, 4, 24), Decimal('153384.691226'), Decimal('17166.039063'))
(datetime.date(2025, 4, 25), Decimal('150793.822639'), Decimal('17382.939453'))
(datetime.date(2025, 4, 28), Decimal('153154.647160'), Decimal('17366.130859'))
(datetime.date(2025, 4, 29), Decimal('153246.471992'), Decimal('17461.320313'))
(datetime.date(2025, 4, 30), Decimal('153045.868306'), Decimal('17446.339844'))
(datetime.date(2025, 5, 1), Decimal('151706.989211'), Decimal('17710.740234'))
(datetime.date(2025, 5, 2), Decimal('1557

In [13]:
#Find days when oil price spiked and compare with Bitcoin price change
cursor.execute("""
SELECT 
    op.date,
    op.price_inr AS oil_price,
    cp.price_inr AS bitcoin_price

FROM oil_prices op
JOIN crypto_prices cp 
ON op.date = cp.date

WHERE cp.coin_id = 'bitcoin'
ORDER BY op.date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 4, 15), Decimal('5124.420000'), Decimal('7271599.950977'))
(datetime.date(2025, 4, 16), Decimal('5219.040000'), Decimal('7171572.949307'))
(datetime.date(2025, 4, 17), Decimal('5400.810000'), Decimal('7201187.297551'))
(datetime.date(2025, 4, 21), Decimal('5268.840000'), Decimal('7263849.726469'))
(datetime.date(2025, 4, 22), Decimal('5361.800000'), Decimal('7447341.987487'))
(datetime.date(2025, 4, 23), Decimal('5199.120000'), Decimal('7975735.892698'))
(datetime.date(2025, 4, 24), Decimal('5274.650000'), Decimal('8003280.213214'))
(datetime.date(2025, 4, 25), Decimal('5299.550000'), Decimal('8000169.783417'))
(datetime.date(2025, 4, 28), Decimal('5253.900000'), Decimal('8009924.381134'))
(datetime.date(2025, 4, 29), Decimal('5132.720000'), Decimal('8092853.390806'))
(datetime.date(2025, 4, 30), Decimal('4942.650000'), Decimal('8027696.597799'))
(datetime.date(2025, 5, 1), Decimal('5028.970000'), Decimal('7968688.759781'))
(datetime.date(2025, 5, 2), Decimal('4952

In [14]:
#Compare top 3 coins daily price trend vs Nifty (^NSEI).
cursor.execute("""
SELECT 
    cp.date,
    cp.coin_id,
    cp.price_inr AS crypto_price,
    sp.close AS nifty_price

FROM crypto_prices cp

-- Join with stock data
JOIN stock_prices sp 
ON cp.date = sp.date

-- Filter top 3 coins
WHERE cp.coin_id IN ('bitcoin', 'ethereum', 'tether')

-- Filter Nifty
AND sp.ticker = '^NSEI'

-- Sort for trend
ORDER BY cp.date, cp.coin_id
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2025, 4, 15), 'bitcoin', Decimal('7271599.950977'), Decimal('23328.550781'))
(datetime.date(2025, 4, 15), 'ethereum', Decimal('139502.269181'), Decimal('23328.550781'))
(datetime.date(2025, 4, 15), 'tether', Decimal('86.024832'), Decimal('23328.550781'))
(datetime.date(2025, 4, 16), 'bitcoin', Decimal('7171572.949307'), Decimal('23437.199219'))
(datetime.date(2025, 4, 16), 'ethereum', Decimal('136118.340300'), Decimal('23437.199219'))
(datetime.date(2025, 4, 16), 'tether', Decimal('85.722129'), Decimal('23437.199219'))
(datetime.date(2025, 4, 17), 'bitcoin', Decimal('7201187.297551'), Decimal('23851.650391'))
(datetime.date(2025, 4, 17), 'ethereum', Decimal('135084.250075'), Decimal('23851.650391'))
(datetime.date(2025, 4, 17), 'tether', Decimal('85.620824'), Decimal('23851.650391'))
(datetime.date(2025, 4, 21), 'bitcoin', Decimal('7263849.726469'), Decimal('24125.550781'))
(datetime.date(2025, 4, 21), 'ethereum', Decimal('135372.221202'), Decimal('24125.550781'))
(datet

In [15]:
#Compare stock prices (^GSPC) with crude oil prices on the same dates
cursor.execute("""
SELECT 
    sp.date,
    sp.close AS sp500_price,
    op.price_inr AS oil_price

FROM stock_prices sp

-- Join oil data using same date
JOIN oil_prices op 
ON sp.date = op.date

-- Filter S&P 500
WHERE sp.ticker = '^GSPC'

-- Sort for trend
ORDER BY sp.date
""")

for row in cursor.fetchall():
    print(row)

(datetime.date(2020, 1, 2), Decimal('3257.850098'), Decimal('5077.110000'))
(datetime.date(2020, 1, 3), Decimal('3234.850098'), Decimal('5229.000000'))
(datetime.date(2020, 1, 6), Decimal('3246.280029'), Decimal('5251.410000'))
(datetime.date(2020, 1, 7), Decimal('3237.179932'), Decimal('5204.100000'))
(datetime.date(2020, 1, 8), Decimal('3253.050049'), Decimal('4950.950000'))
(datetime.date(2020, 1, 9), Decimal('3274.699951'), Decimal('4943.480000'))
(datetime.date(2020, 1, 10), Decimal('3265.350098'), Decimal('4898.660000'))
(datetime.date(2020, 1, 13), Decimal('3288.129883'), Decimal('4828.110000'))
(datetime.date(2020, 1, 14), Decimal('3283.149902'), Decimal('4842.220000'))
(datetime.date(2020, 1, 15), Decimal('3289.290039'), Decimal('4802.380000'))
(datetime.date(2020, 1, 16), Decimal('3316.810059'), Decimal('4857.160000'))
(datetime.date(2020, 1, 17), Decimal('3329.620117'), Decimal('4859.650000'))
(datetime.date(2020, 1, 21), Decimal('3320.790039'), Decimal('4834.750000'))
(date

In [16]:
#Correlate Bitcoin closing price with crude oil closing price (same date)
cursor.execute("""
SELECT 
    cp.date,
    cp.price_inr AS bitcoin_price,
    op.price_inr AS oil_price

FROM crypto_prices cp

-- Join oil data using same date
JOIN oil_prices op 
ON cp.date = op.date

WHERE cp.coin_id = 'bitcoin'

ORDER BY cp.date
""")

data = cursor.fetchall()

for row in data:
    print(row)

(datetime.date(2025, 4, 15), Decimal('7271599.950977'), Decimal('5124.420000'))
(datetime.date(2025, 4, 16), Decimal('7171572.949307'), Decimal('5219.040000'))
(datetime.date(2025, 4, 17), Decimal('7201187.297551'), Decimal('5400.810000'))
(datetime.date(2025, 4, 21), Decimal('7263849.726469'), Decimal('5268.840000'))
(datetime.date(2025, 4, 22), Decimal('7447341.987487'), Decimal('5361.800000'))
(datetime.date(2025, 4, 23), Decimal('7975735.892698'), Decimal('5199.120000'))
(datetime.date(2025, 4, 24), Decimal('8003280.213214'), Decimal('5274.650000'))
(datetime.date(2025, 4, 25), Decimal('8000169.783417'), Decimal('5299.550000'))
(datetime.date(2025, 4, 28), Decimal('8009924.381134'), Decimal('5253.900000'))
(datetime.date(2025, 4, 29), Decimal('8092853.390806'), Decimal('5132.720000'))
(datetime.date(2025, 4, 30), Decimal('8027696.597799'), Decimal('4942.650000'))
(datetime.date(2025, 5, 1), Decimal('7968688.759781'), Decimal('5028.970000'))
(datetime.date(2025, 5, 2), Decimal('8167

In [18]:
#step to corelation
import pandas as pd

# Convert SQL result to DataFrame
df = pd.DataFrame(data, columns=["date", "bitcoin_price", "oil_price"])

# Calculate correlation
correlation = df["bitcoin_price"].corr(df["oil_price"])

print("Correlation:", correlation)

Correlation: -0.4005256992697516


In [19]:
#Compare NASDAQ (^IXIC) with Ethereum price trends
cursor.execute("""
SELECT 
    cp.date,
    cp.price_inr AS ethereum_price,
    sp.close AS nasdaq_price

FROM crypto_prices cp

-- Join stock data using same date
JOIN stock_prices sp 
ON cp.date = sp.date

-- Filter Ethereum
WHERE cp.coin_id = 'ethereum'

-- Filter NASDAQ
AND sp.ticker = '^IXIC'

-- Sort for trend
ORDER BY cp.date
""")

data = cursor.fetchall()

for row in data:
    print(row)

(datetime.date(2025, 4, 15), Decimal('139502.269181'), Decimal('16823.169922'))
(datetime.date(2025, 4, 16), Decimal('136118.340300'), Decimal('16307.160156'))
(datetime.date(2025, 4, 17), Decimal('135084.250075'), Decimal('16286.450195'))
(datetime.date(2025, 4, 21), Decimal('135372.221202'), Decimal('15870.900391'))
(datetime.date(2025, 4, 22), Decimal('134334.008185'), Decimal('16300.419922'))
(datetime.date(2025, 4, 23), Decimal('149984.444802'), Decimal('16708.050781'))
(datetime.date(2025, 4, 24), Decimal('153384.691226'), Decimal('17166.039063'))
(datetime.date(2025, 4, 25), Decimal('150793.822639'), Decimal('17382.939453'))
(datetime.date(2025, 4, 28), Decimal('153154.647160'), Decimal('17366.130859'))
(datetime.date(2025, 4, 29), Decimal('153246.471992'), Decimal('17461.320313'))
(datetime.date(2025, 4, 30), Decimal('153045.868306'), Decimal('17446.339844'))
(datetime.date(2025, 5, 1), Decimal('151706.989211'), Decimal('17710.740234'))
(datetime.date(2025, 5, 2), Decimal('1557

In [20]:
#step to corelation
import pandas as pd

df = pd.DataFrame(data, columns=["date", "ethereum_price", "nasdaq_price"])

correlation = df["ethereum_price"].corr(df["nasdaq_price"])

print("Correlation:", correlation)

Correlation: 0.45183765866000125


In [21]:
#Join top 3 crypto coins with stock indices for 2025
cursor.execute("""
SELECT 
    cp.date,
    cp.coin_id,
    cp.price_inr AS crypto_price,

    sp.ticker,
    sp.close AS stock_price

FROM crypto_prices cp

-- Join with stock data
JOIN stock_prices sp 
ON cp.date = sp.date

-- Top 3 coins
WHERE cp.coin_id IN ('bitcoin', 'ethereum', 'tether')

-- Stock indices
AND sp.ticker IN ('^GSPC', '^IXIC', '^NSEI')

-- Year filter (change if needed)
AND YEAR(cp.date) = 2025

-- Sort nicely
ORDER BY cp.date, cp.coin_id, sp.ticker
""")

data = cursor.fetchall()

for row in data:
    print(row)

(datetime.date(2025, 4, 15), 'bitcoin', Decimal('7271599.950977'), '^GSPC', Decimal('5396.629883'))
(datetime.date(2025, 4, 15), 'bitcoin', Decimal('7271599.950977'), '^IXIC', Decimal('16823.169922'))
(datetime.date(2025, 4, 15), 'bitcoin', Decimal('7271599.950977'), '^NSEI', Decimal('23328.550781'))
(datetime.date(2025, 4, 15), 'ethereum', Decimal('139502.269181'), '^GSPC', Decimal('5396.629883'))
(datetime.date(2025, 4, 15), 'ethereum', Decimal('139502.269181'), '^IXIC', Decimal('16823.169922'))
(datetime.date(2025, 4, 15), 'ethereum', Decimal('139502.269181'), '^NSEI', Decimal('23328.550781'))
(datetime.date(2025, 4, 15), 'tether', Decimal('86.024832'), '^GSPC', Decimal('5396.629883'))
(datetime.date(2025, 4, 15), 'tether', Decimal('86.024832'), '^IXIC', Decimal('16823.169922'))
(datetime.date(2025, 4, 15), 'tether', Decimal('86.024832'), '^NSEI', Decimal('23328.550781'))
(datetime.date(2025, 4, 16), 'bitcoin', Decimal('7171572.949307'), '^GSPC', Decimal('5275.700195'))
(datetime.da

In [22]:
#Multi-join: stock prices, oil prices, and Bitcoin prices for daily comparison
cursor.execute("""
SELECT 
    cp.date,

    -- Bitcoin
    cp.price_inr AS bitcoin_price,

    -- Stock (S&P 500)
    sp.close AS stock_price,

    -- Oil
    op.price_inr AS oil_price

FROM crypto_prices cp

-- Join stock data
JOIN stock_prices sp 
ON cp.date = sp.date

-- Join oil data
JOIN oil_prices op 
ON cp.date = op.date

-- Filter Bitcoin
WHERE cp.coin_id = 'bitcoin'

-- Choose stock index (you can change)
AND sp.ticker = '^GSPC'

-- Sort for daily comparison
ORDER BY cp.date
""")

data = cursor.fetchall()

for row in data:
    print(row)

(datetime.date(2025, 4, 15), Decimal('7271599.950977'), Decimal('5396.629883'), Decimal('5124.420000'))
(datetime.date(2025, 4, 16), Decimal('7171572.949307'), Decimal('5275.700195'), Decimal('5219.040000'))
(datetime.date(2025, 4, 17), Decimal('7201187.297551'), Decimal('5282.700195'), Decimal('5400.810000'))
(datetime.date(2025, 4, 21), Decimal('7263849.726469'), Decimal('5158.200195'), Decimal('5268.840000'))
(datetime.date(2025, 4, 22), Decimal('7447341.987487'), Decimal('5287.759766'), Decimal('5361.800000'))
(datetime.date(2025, 4, 23), Decimal('7975735.892698'), Decimal('5375.859863'), Decimal('5199.120000'))
(datetime.date(2025, 4, 24), Decimal('8003280.213214'), Decimal('5484.770020'), Decimal('5274.650000'))
(datetime.date(2025, 4, 25), Decimal('8000169.783417'), Decimal('5525.209961'), Decimal('5299.550000'))
(datetime.date(2025, 4, 28), Decimal('8009924.381134'), Decimal('5528.750000'), Decimal('5253.900000'))
(datetime.date(2025, 4, 29), Decimal('8092853.390806'), Decimal(

In [ ]:
#📊 Streamlit App Structure

In [3]:
!pip install streamlit

In [2]:
import streamlit as st
!pip install streamlit-option-menu
from streamlit_option_menu import option_menu
import pandas as pd